# Step 7: Locate TCR clonotypes spatially and link them to expression and spatial domain

Reads the integrated and labeled AnnData object (`xenium_integrated_labeled.h5ad`) and uses the in-panel TCR probes (from Step 6.1) to call TCR-positive T cells. For each tracked clone or alpha/beta pair we compute the proportion of TCR-positive T cells per sample, the per-clone distribution across spatial domains, and signature scores for previously published neoantigen-specific T-cell programmes (Caushi, Oliveira, Lowery, Hanada).

CSV outputs are written to `data/processed/`:

- `fig5a.csv` (TCR-positive T-cell proportion per sample, per clone)
- `ext_fig7a_data.csv` and `ext_fig7a_stats.csv` (per-clone counts/proportions per spatial domain and stratified permutation tests)
- `ext_fig7b_data_pt1.csv`, `ext_fig7b_data_pt2.csv` and `ext_fig7b_stats.csv` (per-cell NeoTCR signature scores and Mann-Whitney tests for alpha/beta pairs versus other T cells)

Additional intermediate CSVs (per-cell scores, per-clone-per-domain pseudobulk expression) are saved alongside.


## Setup, imports, paths

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import scipy
import scipy.stats
from scipy import sparse
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests, fdrcorrection

sc.settings.verbosity = 3
np.random.seed(26)

indir   = '/path/to/integrated/processed_data/'
csvdir  = '../data/processed/'
figdir  = '../figures/'
os.makedirs(csvdir, exist_ok=True)
os.makedirs(figdir, exist_ok=True)

plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams['font.family'] = 'Helvetica'
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

## Clone and pair definitions

In [ ]:
# Clones and alpha/beta pairs tracked spatially. Each entry maps a clone
# label (e.g. 'γ Clone 17') to either
#   (clone_prefix, [samples], chain_class)               -- single chain, or
#   (alpha_prefix, beta_prefix, [samples], chain_class)  -- alpha/beta pair.
# `clone_prefix` selects TCR probes from the panel by gene-name prefix
# (e.g. 'clone2_' picks up TRGV.clone2_*); `samples` restricts the analysis
# to patients in which the clone was detected by bulk TCR-seq.
pairs_map = {
    'γ Clone 2':  ('clone2_',  ['Patient2_DX', 'Patient2_PT', 'Patient5_DX'], 'γδ'),
    'γ Clone 4':  ('clone4_',  ['Patient2_PT'], 'γδ'),
    'α Clone 7':  ('clone7_',  ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'γ Clone 15': ('clone15_', ['Patient2_PT'], 'γδ'),
    'γ Clone 17': ('clone17_', ['Patient4_DX', 'Patient1_DX', 'Patient1_PT'], 'γδ'),
    'γ Clone 21': ('clone21_', ['Patient2_DX'], 'γδ'),
    'γ Clone 22': ('clone22_', ['Patient2_DX', 'Patient2_PT'], 'γδ'),
    'δ Clone 28': ('clone28_', ['Patient2_PT'], 'γδ'),
    'δ Clone 33': ('clone33_', ['Patient2_DX', 'Patient2_PT'], 'γδ'),
    'δ Clone 35': ('clone35_', ['Patient2_PT'], 'γδ'),
    'α Clone 39': ('clone39_', ['Patient2_PT'], '⍺β CD4'),
    'α Clone 43': ('clone43_', ['Patient2_PT'], '⍺β CD8'),
    'α Clone 55': ('clone55_', ['Patient5_DX'], '⍺β CD8'),
    'β Clone 65': ('clone65_', ['Patient2_PT'], '⍺β CD8'),
    'β Clone 82': ('clone82_', ['Patient5_PT'], '⍺β CD8'),
    'β Clone 84': ('clone84_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 1':  ('clone45_', 'clone64_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 2':  ('clone52_', 'clone60_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 3':  ('clone51_', 'clone62_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
    'αβ Pair 4':  ('clone57_', 'clone87_', ['Patient2_DX', 'Patient2_PT'], '⍺β CD8'),
}

sample_order = [f'Patient{i}_{tp}' for i in range(1, 6) for tp in ('DX', 'PT')]
categories_order = ['Immune-rich', 'Neuroblast-rich', 'Other']

# Spatial-domain assignment (consistent with Step 5).
immune_rich_clusters     = ['0', '4', '8', '9']
neuroblast_rich_clusters = ['1', '2', '6', '7', '10', '12', '14', '16', '17', '18']
def classify_3_domains(n):
    n_str = str(n)
    if n_str in immune_rich_clusters:     return 'Immune-rich'
    if n_str in neuroblast_rich_clusters: return 'Neuroblast-rich'
    return 'Other'

# Convenience views over pairs_map used by the figure cells below.
clone_labels = list(pairs_map.keys())
pairs = [
    ('clone45_', 'clone64_'),
    ('clone52_', 'clone60_'),
    ('clone51_', 'clone62_'),
    ('clone57_', 'clone87_'),
]
pair_ids = [f'Pair{i+1}' for i in range(len(pairs))]
pair_color, other_color = '#FFA63B', '#E0E0E0'

# Some CSV inputs use ASCII initials for the Greek-letter chain classes; this
# mapping normalises them to the Unicode forms used elsewhere in the notebook.
clone_label_alias = {
    'g Clone 2':  'γ Clone 2',
    'g Clone 17': 'γ Clone 17',
    'g Clone 22': 'γ Clone 22',
    'd Clone 33': 'δ Clone 33',
    'a Clone 7':  'α Clone 7',
    'b Clone 84': 'β Clone 84',
    'ab Pair 1':  'αβ Pair 1',
    'ab Pair 2':  'αβ Pair 2',
    'ab Pair 3':  'αβ Pair 3',
    'ab Pair 4':  'αβ Pair 4',
}

## Clone-mask helpers

In [ ]:
def get_clone_sum(ad, prefix, clone_class=None):
    """Sum probe counts for genes starting with `prefix`; optionally require co-expression of lineage markers."""
    genes = [g for g in ad.var_names if g.startswith(prefix)]
    if not genes:
        return np.zeros(ad.n_obs)
    X = ad[:, genes].X
    clone_sum = np.asarray(X.sum(axis=1)).ravel() if not sparse.issparse(X) else np.asarray(X.sum(axis=1)).ravel()

    required_markers = {
        'γδ':       ['TRGC2', 'CD3E'],
        '⍺β CD8':   ['CD8A', 'CD3E'],
        '⍺β CD4':   ['CD4', 'CD3E'],
    }
    markers = required_markers.get(clone_class, [])
    if not markers:
        return clone_sum
    marker_mask = np.ones(ad.n_obs, dtype=bool)
    for m in markers:
        if m not in ad.var_names:
            return np.zeros(ad.n_obs)
        mx = ad[:, [m]].X
        mvec = np.asarray(mx.toarray()).ravel() if sparse.issparse(mx) else np.asarray(mx).ravel()
        marker_mask &= (mvec > 0)
    return np.where(marker_mask, clone_sum, 0.0)


def get_single_clone_mask(ad, prefix):
    g1 = [g for g in ad.var_names if g.startswith(prefix)]
    if not g1:
        return np.zeros(ad.n_obs, dtype=bool)
    X = ad[:, g1].X
    e1 = np.asarray(X.sum(axis=1)).ravel() if not sparse.issparse(X) else np.asarray(X.sum(axis=1)).ravel()
    return e1 > 0


def get_clone_mask_pair(ad, c1, c2):
    g1 = [g for g in ad.var_names if g.startswith(c1)]
    g2 = [g for g in ad.var_names if g.startswith(c2)]
    if not g1 or not g2:
        return np.zeros(ad.n_obs, dtype=bool)
    e1 = ad[:, g1].X.sum(axis=1); e2 = ad[:, g2].X.sum(axis=1)
    if hasattr(e1, 'A1'): e1 = e1.A1
    if hasattr(e2, 'A1'): e2 = e2.A1
    return (e1 > 0) & (e2 > 0)

## Load h5ad

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')

# Recompute the spatial-domain assignment if it has not been carried over.
if 'Spatial_Domain' not in adata.obs.columns:
    adata.obs['Spatial_Domain'] = adata.obs['neigh_kmeans'].apply(classify_3_domains)

## Heatmap of TCR-positive T-cell proportions per sample (Fig 5a)

In [ ]:
# Per-sample, per-clone TCR-positive T-cell proportion.
samples = sample_order.copy()
prop_df = pd.DataFrame(index=samples, columns=clone_labels, dtype=float)

for sample_id in samples:
    sub = adata[(adata.obs['sample'] == sample_id) & (adata.obs['celltype'] == 'T')].copy()
    n_t = sub.n_obs
    if n_t == 0:
        continue
    for label, vals in pairs_map.items():
        if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
            clone_targets = [vals[0], vals[1]]
            allowed_samples = vals[2]; clone_class = vals[3]
        elif isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
            clone_targets = [vals[0]]
            allowed_samples = vals[1]; clone_class = vals[2]
        else:
            prop_df.loc[sample_id, label] = np.nan; continue

        if sample_id not in allowed_samples:
            prop_df.loc[sample_id, label] = np.nan; continue

        if len(clone_targets) == 1:
            pos_mask = get_clone_sum(sub, clone_targets[0], clone_class) > 0
        else:
            c1 = get_clone_sum(sub, clone_targets[0], clone_class) > 0
            c2 = get_clone_sum(sub, clone_targets[1], clone_class) > 0
            pos_mask = c1 & c2
        prop_df.loc[sample_id, label] = pos_mask.sum() / n_t

keep_mask = prop_df.fillna(0).sum(axis=1) > 0
prop_df = prop_df.loc[keep_mask]

n_t_by_sample = (
    adata.obs.loc[adata.obs['celltype'] == 'T', 'sample']
    .value_counts().reindex(prop_df.index).fillna(0).astype(int)
)
source_long = (prop_df.stack(dropna=False).rename('fraction').reset_index()
               .rename(columns={'level_0': 'sample', 'level_1': 'clone_pair'}))
source_long['n_t_cells']      = source_long['sample'].map(n_t_by_sample)
source_long['count_positive'] = (source_long['fraction'] * source_long['n_t_cells']).round().astype('Int64')
source_long.to_csv(csvdir + 'fig5a.csv', index=False)

In [ ]:
source_long = pd.read_csv(csvdir + 'fig5a.csv')
source_long['clone_pair'] = source_long['clone_pair'].replace(clone_label_alias)

# Pivot to a sample x clone matrix of TCR-positive T-cell proportions and drop
# samples in which none of the tracked clones were detected.
prop_df = (source_long.pivot(index='sample', columns='clone_pair', values='fraction')
                       .reindex(index=sample_order)
                       .reindex(columns=clone_labels))
keep_mask = prop_df.fillna(0).sum(axis=1) > 0
prop_df = prop_df.loc[keep_mask]

# Cell labels show the TCR-positive cell count and proportion.
annot_df = (source_long.assign(
        label=lambda d: d.apply(
            lambda r: '' if pd.isna(r['fraction']) else f"{int(r['count_positive'])}\n({r['fraction']:.3f})",
            axis=1)
    )
    .pivot(index='sample', columns='clone_pair', values='label')
    .reindex(index=prop_df.index)
    .reindex(columns=prop_df.columns))

plt.figure(figsize=(8, 3))
ax = sns.heatmap(
    prop_df,
    cmap='YlGnBu',
    vmin=0,
    vmax=np.nanmax(prop_df.values) if np.isfinite(np.nanmax(prop_df.values)) else 1,
    linewidths=0.5,
    linecolor='white',
    annot=annot_df.fillna(''),
    fmt='',
    cbar_kws={'label': 'Clone count (proportion of T cells)'},
    mask=prop_df.isna(),
)
ax.set_xlabel('Clone / Pair')
ax.set_ylabel('Sample')
plt.xticks(rotation=90, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(figdir + 'fig5a_all_clone_heatmap.pdf', bbox_inches='tight', transparent=True)
plt.show()

## Distribution of TCR-positive T cells across spatial domains (Ext Fig 7a)

For each tracked clone, counts and proportions of TCR-positive T cells in each spatial domain are tabulated per sample, and per-gene differences between domain pairs are evaluated by sample-stratified permutation tests.

In [ ]:
MIN_CELLS_PER_DOMAIN = 10
N_PERM = 10000
RNG_SEED = 42
domain_pairs = [
    ('Immune-rich', 'Neuroblast-rich'),
    ('Immune-rich', 'Other'),
    ('Neuroblast-rich', 'Other'),
]
rng = np.random.default_rng(RNG_SEED)

genes_to_plot = [
    'TCF7', 'LEF1', 'SELL', 'IL7R',
    'THEMIS', 'FOS',
    'IFNG', 'KLRD1', 'GZMH', 'GZMB',
    'TOX', 'LAG3',
]

def _to_dense(X):
    return X.toarray() if sparse.issparse(X) else np.asarray(X)

def _get_entry_info(vals):
    if isinstance(vals, tuple) and len(vals) == 3 and isinstance(vals[1], list):
        return {'entry_type': 'clone', 'c1': vals[0], 'c2': None, 'samples': vals[1], 'tcr_type': vals[2]}
    if isinstance(vals, tuple) and len(vals) == 4 and isinstance(vals[2], list):
        return {'entry_type': 'pair',  'c1': vals[0], 'c2': vals[1], 'samples': vals[2], 'tcr_type': vals[3]}
    return None

def _obs_stat(df, d1, d2):
    sample_deltas = []
    for s, g in df.groupby('sample'):
        m1 = g.loc[g['domain'] == d1, 'expr'].mean()
        m2 = g.loc[g['domain'] == d2, 'expr'].mean()
        sample_deltas.append(m2 - m1)
    return float(np.mean(sample_deltas)), np.array(sample_deltas, dtype=float)

def _stratified_perm(cell_df, d1, d2, n_perm=10000, rng=None):
    if rng is None:
        rng = np.random.default_rng(1)
    obs_stat, sample_deltas = _obs_stat(cell_df, d1, d2)
    by_sample = []
    for s, g in cell_df.groupby('sample'):
        by_sample.append((s, g['domain'].to_numpy(), g['expr'].to_numpy()))
    perm_stats = np.empty(n_perm, dtype=float)
    for i in range(n_perm):
        deltas_i = []
        for _, dom, expr in by_sample:
            perm_dom = rng.permutation(dom)
            m1 = expr[perm_dom == d1].mean()
            m2 = expr[perm_dom == d2].mean()
            deltas_i.append(m2 - m1)
        perm_stats[i] = np.mean(deltas_i)
    p_two = (np.sum(np.abs(perm_stats) >= np.abs(obs_stat)) + 1.0) / (n_perm + 1.0)
    return obs_stat, p_two, sample_deltas

In [ ]:
count_rows = []
stat_rows  = []

for label, vals in pairs_map.items():
    info = _get_entry_info(vals)
    if info is None:
        continue

    sub_all = adata[(adata.obs['sample'].isin(info['samples'])) & (adata.obs['celltype'] == 'T')].copy()
    if sub_all.n_obs == 0:
        continue
    mask = (get_single_clone_mask(sub_all, info['c1'])
            if info['entry_type'] == 'clone'
            else get_clone_mask_pair(sub_all, info['c1'], info['c2']))
    sub = sub_all[mask].copy()
    if sub.n_obs == 0:
        continue
    sub.obs['domain'] = sub.obs['neigh_kmeans'].map(classify_3_domains)
    sub = sub[sub.obs['domain'].isin(categories_order)].copy()
    if sub.n_obs == 0:
        continue

    ctab = sub.obs.groupby(['sample', 'domain']).size().reset_index(name='n_clonepos_cells')
    ctot = sub.obs.groupby('sample').size().reset_index(name='n_clonepos_total')
    ctab = ctab.merge(ctot, on='sample', how='left')
    ctab['prop_within_clonepos'] = ctab['n_clonepos_cells'] / ctab['n_clonepos_total']
    ctab['label']      = label
    ctab['entry_type'] = info['entry_type']
    ctab['tcr_type']   = info['tcr_type']
    count_rows.append(ctab)

    genes = [g for g in genes_to_plot if g in sub.var_names]
    if len(genes) == 0:
        continue
    X = _to_dense(sub[:, genes].X)
    expr_df = pd.DataFrame(X, columns=genes, index=sub.obs_names)
    long_df = sub.obs[['sample', 'domain']].copy().join(expr_df).reset_index(drop=True).melt(
        id_vars=['sample', 'domain'], var_name='gene', value_name='expr'
    )

    for g in genes:
        dg = long_df[long_df['gene'] == g]
        for d1, d2 in domain_pairs:
            dsub = dg[dg['domain'].isin([d1, d2])].copy()
            if dsub.empty:
                continue
            keep_samples = []
            for s, gs in dsub.groupby('sample'):
                n1 = int((gs['domain'] == d1).sum()); n2 = int((gs['domain'] == d2).sum())
                if n1 >= MIN_CELLS_PER_DOMAIN and n2 >= MIN_CELLS_PER_DOMAIN:
                    keep_samples.append(s)
            dsub = dsub[dsub['sample'].isin(keep_samples)]
            n_samples = dsub['sample'].nunique()
            row = {'label': label, 'entry_type': info['entry_type'], 'tcr_type': info['tcr_type'],
                   'gene': g, 'domain_1': d1, 'domain_2': d2,
                   'n_samples_with_both_domains': int(n_samples),
                   'min_cells_per_domain_rule': int(MIN_CELLS_PER_DOMAIN),
                   'obs_mean_delta_d2_minus_d1': np.nan,
                   'median_sample_delta_d2_minus_d1': np.nan,
                   'perm_p_two_sided': np.nan,
                   'inference_tier': 'not_tested'}
            if n_samples >= 2:
                obs_d, p_perm, sample_deltas = _stratified_perm(dsub[['sample','domain','expr']],
                                                                d1, d2, n_perm=N_PERM, rng=rng)
                row['obs_mean_delta_d2_minus_d1'] = float(obs_d)
                row['median_sample_delta_d2_minus_d1'] = float(np.median(sample_deltas))
                row['perm_p_two_sided'] = float(p_perm)
                row['inference_tier'] = 'exploratory_n2' if n_samples == 2 else 'sample_level'
            stat_rows.append(row)

counts_df = pd.concat(count_rows, ignore_index=True) if count_rows else pd.DataFrame()
stats_df  = pd.DataFrame(stat_rows)

if not stats_df.empty:
    valid = stats_df['perm_p_two_sided'].notna()
    stats_df['qval_bh_global'] = np.nan
    if valid.any():
        stats_df.loc[valid, 'qval_bh_global'] = multipletests(stats_df.loc[valid, 'perm_p_two_sided'].values, method='fdr_bh')[1]
    stats_df['qval_bh_within_label'] = np.nan
    for lb, idx in stats_df.groupby('label').groups.items():
        idx = np.array(list(idx))
        m = stats_df.loc[idx, 'perm_p_two_sided'].notna().values
        if m.any():
            stats_df.loc[idx[m], 'qval_bh_within_label'] = multipletests(
                stats_df.loc[idx[m], 'perm_p_two_sided'].values, method='fdr_bh')[1]
    def q_to_stars(q):
        if pd.isna(q): return ''
        if q < 1e-3:   return '***'
        if q < 1e-2:   return '**'
        if q < 5e-2:   return '*'
        return ''
    stats_df['stars_within_label'] = stats_df['qval_bh_within_label'].apply(q_to_stars)

if not counts_df.empty:
    counts_df.to_csv(csvdir + 'ext_fig7a_data.csv', index=False)
if not stats_df.empty:
    stats_df.to_csv(csvdir + 'ext_fig7a_stats.csv', index=False)

display(counts_df.head()); display(stats_df.head())

## NeoTCR signature scoring of alpha/beta pairs in Patient2 (Ext Fig 7b)

Per-cell signature scores (Caushi, Oliveira, Lowery, Hanada) are computed with `sc.tl.score_genes`. For each tracked alpha/beta pair, signature scores in pair-positive T cells are compared against the remaining T cells with a Mann-Whitney U test, and p-values are Benjamini-Hochberg adjusted across pairs, timepoints and signatures.

In [ ]:
gl_df = pd.read_csv(csvdir + 'suppl_table_4_filtered_gene_lists.csv')
signatures = {col: [g for g in gl_df[col].dropna().astype(str).tolist() if g] for col in gl_df.columns}
print({k: len(v) for k, v in signatures.items()})

# αβ pairs were only detected in Patient2.
samples = ['Patient2_DX', 'Patient2_PT']

# Compute scores
for name, genes in signatures.items():
    valid_genes = [g for g in genes if g in adata.var_names]
    if valid_genes:
        sc.tl.score_genes(adata, gene_list=valid_genes, score_name=f'score_{name}')

# Stats: Mann-Whitney Pair vs Other, BH FDR
results_data = []
for sample in samples:
    sub = adata[(adata.obs['sample'] == sample) & (adata.obs['celltype'] == 'T')].copy()
    for c1, c2 in pairs:
        is_pair = get_clone_mask_pair(sub, c1, c2)
        n_pair = int(is_pair.sum()); n_other = int(len(is_pair) - n_pair)
        if n_pair < 3 or n_other < 3:
            for sig_name in signatures:
                results_data.append({'sample': sample, 'pair': f'{c1}+{c2}',
                                     'signature': sig_name, 'p': np.nan})
            continue
        for sig_name in signatures:
            sc_col = f'score_{sig_name}'
            s_pair  = sub.obs.loc[is_pair, sc_col]
            s_other = sub.obs.loc[~is_pair, sc_col]
            _, p = mannwhitneyu(s_pair, s_other, alternative='two-sided')
            results_data.append({'sample': sample, 'pair': f'{c1}+{c2}',
                                 'signature': sig_name, 'p': float(p)})
results_df = pd.DataFrame(results_data)
valid = results_df['p'].notna()
if valid.any():
    reject, q = fdrcorrection(results_df.loc[valid, 'p'].values, alpha=0.05)
    results_df.loc[valid, 'q'] = q
    results_df.loc[valid, 'significant_fdr'] = reject
def q_to_stars(qv):
    if pd.isna(qv): return ''
    if qv < 1e-3:   return '***'
    if qv < 1e-2:   return '**'
    if qv < 5e-2:   return '*'
    return ''
results_df['stars'] = results_df['q'].apply(q_to_stars)

pair_label_map = {pair_ids[i]: f'{pairs[i][0]}+{pairs[i][1]}' for i in range(len(pairs))}
pair_to_id = {v: k for k, v in pair_label_map.items()}

def sample_to_timepoint(s):
    s = str(s)
    if s.endswith('_DX'): return 'DX'
    if s.endswith('_PT'): return 'PT'
    return np.nan

# Build long plot_df (Pair / Other cells per signature)
rows = []
for sample in samples:
    sub = adata[(adata.obs['sample'] == sample) & (adata.obs['celltype'] == 'T')].copy()
    if sub.n_obs == 0:
        continue
    tp = sample_to_timepoint(sample)
    for c1, c2 in pairs:
        pair_label = f'{c1}+{c2}'; pid = pair_to_id[pair_label]
        is_pair = get_clone_mask_pair(sub, c1, c2)
        if is_pair.sum() < 3 or (~is_pair).sum() < 3:
            continue
        for sig in signatures:
            sc_col = f'score_{sig}'
            if sc_col not in sub.obs.columns:
                continue
            for v in sub.obs.loc[is_pair, sc_col].values:
                rows.append({'sample': sample, 'timepoint': tp, 'pair': pair_label,
                             'pair_id': pid, 'signature': sig, 'group': 'Pair', 'score': float(v)})
            for v in sub.obs.loc[~is_pair, sc_col].values:
                rows.append({'sample': sample, 'timepoint': tp, 'pair': pair_label,
                             'pair_id': pid, 'signature': sig, 'group': 'Other', 'score': float(v)})
plot_df = pd.DataFrame(rows)
tests_df = results_df.copy()
tests_df['timepoint'] = tests_df['sample'].apply(sample_to_timepoint)
tests_df['pair_id'] = tests_df['pair'].map(pair_to_id)

# Save plot data split into halves (matches '7b data pt1/pt2' format)
mid = (len(plot_df) + 1) // 2
plot_df.iloc[:mid].to_csv(csvdir + 'ext_fig7b_data_pt1.csv', index=True)
plot_df.iloc[mid:].to_csv(csvdir + 'ext_fig7b_data_pt2.csv', index=True)
tests_df.to_csv(csvdir + 'ext_fig7b_stats.csv', index=True)

In [ ]:
plot_df  = pd.concat([pd.read_csv(csvdir + 'ext_fig7b_data_pt1.csv', index_col=0),
                      pd.read_csv(csvdir + 'ext_fig7b_data_pt2.csv', index_col=0)], ignore_index=True)
tests_df = pd.read_csv(csvdir + 'ext_fig7b_stats.csv', index_col=0)

FONT_SIZE = 6
sig_order = plot_df['signature'].dropna().drop_duplicates().tolist()
tp_present = plot_df['timepoint'].dropna().unique().tolist()
tp_order = [t for t in ['DX', 'PT'] if t in tp_present] + [t for t in tp_present if t not in ['DX', 'PT']]

score_vals = plot_df['score'].dropna().values
if len(score_vals) == 0:
    y_lower, y_upper = 0.0, 1.0
else:
    y_lower = float(np.min(score_vals)); y_upper = float(np.max(score_vals))
    pad = 0.05 * (y_upper - y_lower if y_upper > y_lower else 1.0)
    y_lower -= pad; y_upper += pad

fig, axes = plt.subplots(nrows=len(tp_order), ncols=len(sig_order),
                         figsize=(13/2.54, 5/2.54), squeeze=False)
for r, tp in enumerate(tp_order):
    for c, sig in enumerate(sig_order):
        ax = axes[r, c]
        d = plot_df[(plot_df['timepoint'] == tp) & (plot_df['signature'] == sig)]
        if d.empty:
            ax.axis('off'); continue
        sns.boxplot(data=d, x='pair_id', y='score', hue='group',
                    order=pair_ids, hue_order=['Pair', 'Other'],
                    palette={'Pair': pair_color, 'Other': other_color},
                    showfliers=False, ax=ax,
                    linewidth=0.5, boxprops={'linewidth': 0.5},
                    whiskerprops={'linewidth': 0.5}, capprops={'linewidth': 0.5},
                    medianprops={'linewidth': 0.5})
        ax.set_ylim(y_lower, y_upper)
        y_text = y_upper - 0.03 * (y_upper - y_lower)
        for i_pid, pid in enumerate(pair_ids):
            row = tests_df[(tests_df['timepoint'] == tp) & (tests_df['signature'] == sig) &
                           (tests_df['pair_id'] == pid)]
            if len(row) == 1:
                star = row['stars'].iloc[0]
                if star:
                    ax.text(i_pid, y_text, star, ha='center', va='top',
                            fontsize=FONT_SIZE, fontweight='bold')
        if r != len(tp_order) - 1:
            ax.set_xticklabels([]); ax.set_xlabel('')
            ax.set_title(sig, fontsize=FONT_SIZE)
        else:
            ax.set_xticklabels(pair_ids, rotation=0, fontsize=FONT_SIZE)
            ax.set_xlabel(''); ax.set_title('')
        ax.set_ylabel('Score' if c == 0 else '')
        ax.tick_params(axis='both', which='both', width=0.5, length=2, labelsize=FONT_SIZE)
        ax.grid(False)
        for side in ['top','right','left','bottom']:
            ax.spines[side].set_visible(False)
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()
plt.tight_layout()
plt.savefig(figdir + 'ext_fig7b_neotcr_2x4.pdf', format='pdf', bbox_inches='tight')
plt.show()

## Spatial maps of individual clones

In [ ]:
for clone_name, vals in pairs_map.items():
    if len(vals) == 3:
        clone_targets = [vals[0]]; target_samples = vals[1]; tcr_type = vals[2]
    else:
        clone_targets = [vals[0], vals[1]]; target_samples = vals[2]; tcr_type = vals[3]

    n = len(target_samples)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
    axes = np.atleast_1d(axes).flatten()
    for i, sample in enumerate(target_samples):
        ax = axes[i]
        sub = adata[adata.obs['sample'] == sample].copy()
        if sub.n_obs == 0:
            ax.axis('off'); continue
        is_target = sub.obs['celltype'] == 'T'
        if len(clone_targets) == 1:
            clone_mask = get_clone_sum(sub, clone_targets[0], tcr_type) > 0
            clone_label = clone_targets[0]
        else:
            c1 = get_clone_sum(sub, clone_targets[0], tcr_type) > 0
            c2 = get_clone_sum(sub, clone_targets[1], tcr_type) > 0
            clone_mask = c1 & c2
            clone_label = f'{clone_targets[0]} & {clone_targets[1]}'
        is_dp = clone_mask & is_target
        x = sub.obsm['spatial'][:, 0]; y = sub.obsm['spatial'][:, 1]
        ax.scatter(x[~is_target], y[~is_target], c='#d3d3d3', s=2, alpha=0.8, linewidths=0)
        ax.scatter(x[is_target],  y[is_target],  c='#74add1', s=2, alpha=0.8, linewidths=0)
        ax.scatter(x[is_dp],      y[is_dp],      c='#000000', s=15, alpha=1, linewidths=0)
        ax.set_title(f'{sample}\n{clone_label}+ (n={int(is_dp.sum())})')
        ax.axis('equal'); ax.axis('off')
    plt.tight_layout()
    plt.savefig(figdir + f'spatial_{clone_name}.png', format='png', bbox_inches='tight', transparent=True, dpi=600)
    plt.show(); plt.close(fig)

## Dotplots of T-cell phenotype by spatial domain for each clone

In [ ]:
for label, vals in pairs_map.items():
    info = _get_entry_info(vals)
    if info is None:
        continue
    sub_all = adata[(adata.obs['sample'].isin(info['samples'])) & (adata.obs['celltype'] == 'T')].copy()
    if sub_all.n_obs == 0:
        continue
    mask = (get_single_clone_mask(sub_all, info['c1'])
            if info['entry_type'] == 'clone'
            else get_clone_mask_pair(sub_all, info['c1'], info['c2']))
    sub_all = sub_all[mask].copy()
    if sub_all.n_obs == 0:
        continue
    sub_all.obs['domain'] = sub_all.obs['neigh_kmeans'].map(classify_3_domains)
    sub_all = sub_all[sub_all.obs['domain'].notna()].copy()
    if sub_all.n_obs == 0:
        continue
    present = sub_all.obs['domain'].value_counts()
    cats_present = [c for c in categories_order if c in present.index]
    if len(cats_present) < 2:
        continue
    genes = [g for g in genes_to_plot if g in sub_all.var_names]
    if not genes:
        continue
    dp = sc.pl.dotplot(sub_all, genes, groupby='domain',
                        categories_order=cats_present, cmap='YlOrRd',
                        return_fig=True, show=False, standard_scale='var')
    dp = dp.style(smallest_dot=0.01, largest_dot=70)
    fig = dp.fig; fig.set_size_inches(10/2.54, 3.5/2.54, forward=True)
    fig.suptitle(f'{label} ({info["tcr_type"]})', fontsize=6, y=1.02)
    fig.tight_layout()
    fig.savefig(figdir + f'dotplot_{label}_domains.pdf', bbox_inches='tight')
    plt.close(fig)